In [5]:
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder
from sklearn.pipeline import Pipeline
from scikeras.wrappers import KerasClassifier
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping
import pickle


In [6]:
data=pd.read_csv('Churn_Modelling.csv')
data.head()

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [7]:
data=data.drop(['RowNumber', 'CustomerId', 'Surname'], axis=1)
label_encoder_gender=LabelEncoder()
data['Gender']=label_encoder_gender.fit_transform(data['Gender'])

onehotencoder_geo=OneHotEncoder(handle_unknown='ignore')
geo_encoded=onehotencoder_geo.fit_transform(data[['Geography']]).toarray()
geo_encoded_df=pd.DataFrame(geo_encoded, columns=onehotencoder_geo.get_feature_names_out(['Geography']))

data=pd.concat([data.drop("Geography", axis=1), geo_encoded_df], axis=1)

X= data.drop('Exited', axis = 1)
y = data['Exited']

X_train, X_test, y_train, y_test= train_test_split(X, y, test_size=0.2, random_state=42)

scaler=StandardScaler()
X_train=scaler.fit_transform(X_train)
X_test=scaler.transform(X_test)

with open('onehotencoder_geo.pkl', 'wb') as file:
    pickle.dump(onehotencoder_geo, file)

with open('label_encoder_gender.pkl', 'wb') as file:
    pickle.dump(label_encoder_gender, file)

with open('scaler.pkl', 'wb') as file:
    pickle.dump(scaler, file)

In [23]:
##Creating a model to try different parameters(KerasClassifier)
def create_model(neuron=32, layers=1):
    model=Sequential()
    model.add(Dense(neuron, activation='relu', input_shape=(X_train.shape[1],)))

    for _ in range(layers - 1 ):
        model.add(Dense(neuron, activation='relu'))

    model.add(Dense(1, activation='sigmoid'))
    model.compile(optimizer='adam', loss="binary_crossentropy", metrics=['accuracy'])

    return model    

In [24]:
##creating keras classifier
model=KerasClassifier(layers=1, neuron=32, build_fn=create_model, epochs=50, batch_size=10, verbose=1)

In [25]:
##Defining GridSearch params
param_grid = {
    'neuron': [16, 32, 64, 128],
    'layers': [1, 2, 3],
    'epochs': [50, 100]
}

In [26]:
##performing grid search
grid = GridSearchCV(estimator=model, param_grid=param_grid, n_jobs=-1, cv=3)
grid_result=grid.fit(X_train, y_train)

##printing the best params
print("Best: %f using %s" % (grid_result.best_score_, grid_result.best_params_))

/Users/sandeshsaidapur/ML Learning projects/ANNClassification/venv/lib/python3.11/site-packages/scikeras/wrappers.py:925: UserWarning: ``build_fn`` will be renamed to ``model`` in a future release, at which point use of ``build_fn`` will raise an Error instead.
  X, y = self._initialize(X, y)
/Users/sandeshsaidapur/ML Learning projects/ANNClassification/venv/lib/python3.11/site-packages/keras/src/layers/core/dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
/Users/sandeshsaidapur/ML Learning projects/ANNClassification/venv/lib/python3.11/site-packages/scikeras/wrappers.py:925: UserWarning: ``build_fn`` will be renamed to ``model`` in a future release, at which point use of ``build_fn`` will raise an Error instead.
  X, y = self._initialize(X, y)
/Users/sandeshsaidapur/M

Epoch 1/50
Epoch 1/50
Epoch 1/50
Epoch 1/50
Epoch 1/50
Epoch 1/50
Epoch 1/50
Epoch 1/50
Epoch 1/50
Epoch 1/50
534/534 ━━━━━━━━━━━━━━━━━━━━ 1s 621us/step - accuracy: 0.7969 - loss: 0.4691
Epoch 2/50
534/534 ━━━━━━━━━━━━━━━━━━━━ 1s 922us/step - accuracy: 0.7685 - loss: 0.5085
Epoch 2/50
534/534 ━━━━━━━━━━━━━━━━━━━━ 1s 924us/step - accuracy: 0.7600 - loss: 0.5311
Epoch 2/50
534/534 ━━━━━━━━━━━━━━━━━━━━ 1s 904us/step - accuracy: 0.7816 - loss: 0.4728
Epoch 2/50
534/534 ━━━━━━━━━━━━━━━━━━━━ 1s 907us/step - accuracy: 0.7493 - loss: 0.5110
534/534 ━━━━━━━━━━━━━━━━━━━━ 1s 892us/step - accuracy: 0.8024 - loss: 0.4592
Epoch 2/50
506/534 ━━━━━━━━━━━━━━━━━━━━ 0s 898us/step - accuracy: 0.7968 - loss: 0.4672Epoch 2/50
534/534 ━━━━━━━━━━━━━━━━━━━━ 1s 901us/step - accuracy: 0.8095 - loss: 0.4424
Epoch 2/50
534/534 ━━━━━━━━━━━━━━━━━━━━ 1s 916us/step - accuracy: 0.7995 - loss: 0.4627
Epoch 2/50
534/534 ━━━━━━━━━━━━━━━━━━━━ 1s 968us/step - accuracy: 0.7684 - loss: 0.4990
Epoch 2/50
534/534 ━━━━━━━━━━━━━━

/Users/sandeshsaidapur/ML Learning projects/ANNClassification/venv/lib/python3.11/site-packages/scikeras/wrappers.py:925: UserWarning: ``build_fn`` will be renamed to ``model`` in a future release, at which point use of ``build_fn`` will raise an Error instead.
  X, y = self._initialize(X, y)
/Users/sandeshsaidapur/ML Learning projects/ANNClassification/venv/lib/python3.11/site-packages/keras/src/layers/core/dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
/Users/sandeshsaidapur/ML Learning projects/ANNClassification/venv/lib/python3.11/site-packages/scikeras/wrappers.py:925: UserWarning: ``build_fn`` will be renamed to ``model`` in a future release, at which point use of ``build_fn`` will raise an Error instead.
  X, y = self._initialize(X, y)
/Users/sandeshsaidapur/M

267/267 ━━━━━━━━━━━━━━━━━━━━ 0s 695us/step - accuracy: 0.8676 - loss: 0.310
534/534 ━━━━━━━━━━━━━━━━━━━━ 1s 935us/step - accuracy: 0.8671 - loss: 0.3259
Epoch 50/50
 57/534 ━━━━━━━━━━━━━━━━━━━━ 0s 907us/step - accuracy: 0.8649 - loss: 0.3258  Epoch 1/50


/Users/sandeshsaidapur/ML Learning projects/ANNClassification/venv/lib/python3.11/site-packages/scikeras/wrappers.py:925: UserWarning: ``build_fn`` will be renamed to ``model`` in a future release, at which point use of ``build_fn`` will raise an Error instead.
  X, y = self._initialize(X, y)
/Users/sandeshsaidapur/ML Learning projects/ANNClassification/venv/lib/python3.11/site-packages/keras/src/layers/core/dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


...

534/534 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.9295 - loss: 0.1564
Epoch 26/100
534/534 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.9229 - loss: 0.1828
534/534 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.9340 - loss: 0.1619
Epoch 76/100
Epoch 43/100
279/534 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.8932 - loss: 0.2596Epoch 1/100
534/534 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.8806 - loss: 0.2771
370/534 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.9330 - loss: 0.1700Epoch 97/100
534/534 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.9160 - loss: 0.1936
Epoch 34/100
534/534 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.9312 - loss: 0.1674
Epoch 83/100
534/534 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.9310 - loss: 0.1749
Epoch 86/100
534/534 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.8903 - loss: 0.2671  
Epoch 99/100
534/534 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.9623 - loss: 0.0958
Epoch 56/100
534/534 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - acc

/Users/sandeshsaidapur/ML Learning projects/ANNClassification/venv/lib/python3.11/site-packages/scikeras/wrappers.py:925: UserWarning: ``build_fn`` will be renamed to ``model`` in a future release, at which point use of ``build_fn`` will raise an Error instead.
  X, y = self._initialize(X, y)
/Users/sandeshsaidapur/ML Learning projects/ANNClassification/venv/lib/python3.11/site-packages/keras/src/layers/core/dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


534/534 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.9282 - loss: 0.1716 
Epoch 89/100
534/534 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.9623 - loss: 0.0997
Epoch 59/100
380/534 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9453 - loss: 0.1367Epoch 1/100
534/534 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.9421 - loss: 0.1360
Epoch 29/100
534/534 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.8241 - loss: 0.4179
Epoch 2/100
534/534 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.9394 - loss: 0.1458
Epoch 47/100
534/534 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.9287 - loss: 0.1634
Epoch 86/100
534/534 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.9226 - loss: 0.1793
Epoch 80/100
534/534 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.8809 - loss: 0.2735  
534/534 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.9229 - loss: 0.1748  
Epoch 38/100
534/534 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.9634 - loss: 0.0898
Epoch 60/100
267/267 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - 

/Users/sandeshsaidapur/ML Learning projects/ANNClassification/venv/lib/python3.11/site-packages/scikeras/wrappers.py:925: UserWarning: ``build_fn`` will be renamed to ``model`` in a future release, at which point use of ``build_fn`` will raise an Error instead.
  X, y = self._initialize(X, y)
/Users/sandeshsaidapur/ML Learning projects/ANNClassification/venv/lib/python3.11/site-packages/keras/src/layers/core/dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


800/800 ━━━━━━━━━━━━━━━━━━━━ 1s 265us/step - accuracy: 0.8081 - loss: 0.4503
Epoch 2/50
800/800 ━━━━━━━━━━━━━━━━━━━━ 0s 252us/step - accuracy: 0.8322 - loss: 0.4005
Epoch 3/50
800/800 ━━━━━━━━━━━━━━━━━━━━ 0s 254us/step - accuracy: 0.8466 - loss: 0.3749
Epoch 4/50
800/800 ━━━━━━━━━━━━━━━━━━━━ 0s 250us/step - accuracy: 0.8529 - loss: 0.3592
Epoch 5/50
800/800 ━━━━━━━━━━━━━━━━━━━━ 0s 251us/step - accuracy: 0.8561 - loss: 0.3517
Epoch 6/50
800/800 ━━━━━━━━━━━━━━━━━━━━ 0s 251us/step - accuracy: 0.8555 - loss: 0.3475
Epoch 7/50
800/800 ━━━━━━━━━━━━━━━━━━━━ 0s 252us/step - accuracy: 0.8579 - loss: 0.3449
Epoch 8/50
800/800 ━━━━━━━━━━━━━━━━━━━━ 0s 339us/step - accuracy: 0.8596 - loss: 0.3428
Epoch 9/50
800/800 ━━━━━━━━━━━━━━━━━━━━ 0s 251us/step - accuracy: 0.8605 - loss: 0.3407
Epoch 10/50
800/800 ━━━━━━━━━━━━━━━━━━━━ 0s 261us/step - accuracy: 0.8597 - loss: 0.3405
Epoch 11/50
800/800 ━━━━━━━━━━━━━━━━━━━━ 0s 252us/step - accuracy: 0.8614 - loss: 0.3398
Epoch 12/50
800/800 ━━━━━━━━━━━━━━━━━━━━ 